# Diffusion Foundations for Medical Imaging

## From DDPM Forward Noising to DDIM Fast Sampling


### Overview

Diffusion models are a family of generative models that learn to recover structured data from noise. In image generation, they define a **forward process** that gradually corrupts a clean image with Gaussian noise and a **reverse process** that learns to remove this noise step by step.

This tutorial introduces the foundations of two influential diffusion methods:

* **Denoising Diffusion Probabilistic Models (DDPM)**
  Ho, Jain, and Abbeel, 2020

* **Denoising Diffusion Implicit Models (DDIM)**
  Song, Meng, and Ermon, 2022

Rather than training a large medical-image generation model from scratch, this notebook focuses on the core mathematical and computational mechanisms behind diffusion models. You will implement the forward noising process, examine how a neural network learns to predict noise, recover estimates of clean images, and construct a faster DDIM sampling procedure.

A 2-D medical image will be used throughout the notebook so that each step can be interpreted in the context of medical imaging.


The central questions are:

* How is a clean image gradually converted into Gaussian noise?
* Why can any noisy state (x_t) be sampled directly from the original image (x_0)?
* What does the neural network actually learn to predict?
* How is a clean-image estimate recovered from a noisy image?
* Why does standard DDPM sampling require many sequential steps?
* How can DDIM reuse the same trained model while sampling with fewer steps?


## Core Idea

The diffusion pipeline can be summarized as:

```text
Clean medical image x₀
        │
        │  Forward diffusion
        │  Gradually add Gaussian noise
        ▼
Noisy image xₜ
        │
        │  Noise-prediction model εθ(xₜ, t)
        ▼
Estimated noise ε̂
        │
        │  Reverse DDPM or DDIM update
        ▼
Recovered or generated image
```

The forward process is fixed and does not need to be learned.

The difficult part is the reverse process: a neural network must estimate the noise contained in an image at every diffusion timestep.


## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain the roles of $\beta_t$, $\alpha_t$, and $\bar{\alpha}_t$ in a diffusion process.

2. Construct a linear noise schedule following the original DDPM formulation.

3. Implement the closed-form DDPM forward process:

$$
x_t
=
\sqrt{\bar{\alpha}_t}\,x_0
+
\sqrt{1-\bar{\alpha}_t}\,\epsilon,
\qquad
\epsilon \sim \mathcal{N}(0,I)
$$

4. Visualize how anatomical structures disappear as the noise level increases.

5. Explain why DDPM trains a neural network to predict the added noise:

$$
\epsilon_\theta(x_t,t)
$$

6. Implement the simplified DDPM noise-prediction objective:

$$
\mathcal{L}_{\mathrm{simple}}
=
\mathbb{E}
\left[
\left\|
\epsilon
-
\epsilon_\theta(x_t,t)
\right\|_2^2
\right]
$$

7. Recover an estimate of the clean image from a noisy image and a predicted noise tensor:

$$
\hat{x}_0
=
\frac{
x_t
-
\sqrt{1-\bar{\alpha}_t}\,
\epsilon_\theta(x_t,t)
}{
\sqrt{\bar{\alpha}_t}
}
$$

8. Explain the difference between stochastic DDPM sampling and deterministic DDIM sampling.

9. Implement a reduced DDIM sampling trajectory using only a subset of the original diffusion timesteps.

10. Analyze the trade-off between sampling speed and reconstruction quality.

11. Explain how DDIM provides a conceptual connection between discrete diffusion models and ODE-based generative models such as Flow Matching.


## Medical Imaging Context

Diffusion models have been investigated for applications such as:

* MRI reconstruction;
* low-dose CT denoising;
* medical-image super-resolution;
* missing-modality synthesis;
* image inpainting;
* synthetic data augmentation.


## References

1. Ho, J., Jain, A., and Abbeel, P.
   *Denoising Diffusion Probabilistic Models.*
   Advances in Neural Information Processing Systems, 2020.

2. Song, J., Meng, C., and Ermon, S.
   *Denoising Diffusion Implicit Models.*
   International Conference on Learning Representations, 2022.


# Part 1 — Preparing the MRI Reference Case

Before studying the diffusion process, we first need to prepare a clean reference image.

In this notebook, we will use the same axial T2-weighted brain MRI slice throughout all experiments. The image will serve as the clean reference image, denoted by (x_0).

Our goal in this section is to convert the downloaded MRI image into a diffusion-ready PyTorch tensor.

The final tensor must satisfy the following requirements:

* grayscale image with one channel;
* spatial resolution of (128 \times 128);
* tensor shape ([B,C,H,W]);
* floating-point data type;
* intensity values normalized to ([-1,1]).

The resulting tensor will have the form:

$$
[
x_0 \in \mathbb{R}^{1 \times 1 \times 128 \times 128}
]
$$

where:

* (B=1) is the batch size;
* (C=1) is the grayscale channel;
* (H=W=128) are the spatial dimensions.

The original DDPM implementation scales image values to ([-1,1]), allowing the reverse model to operate on consistently scaled inputs while the final diffusion state approaches a standard Gaussian distribution.



## Task Scenario

Imaging that you are preparing a reference axial T2-weighted brain MRI for a controlled diffusion reconstruction experiment.

Before Gaussian noise can be added, you must:

1. download the MRI from a public URL;
2. verify that the image is readable;
3. convert it to grayscale;
4. crop it to a square region without significantly distorting the anatomy;
5. resize it to (128 \times 128);
6. convert it into a PyTorch tensor;
7. normalize its intensity values to ([-1,1]).

This prepared image will be stored as `x0` and reused in every later part of the notebook.


In [ ]:
# Part 1.1 — Environment setup

import random
from io import BytesIO
from urllib.request import Request, urlopen

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import time

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Runtime device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Environment setup completed.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

## Loading the Reference MRI

The MRI image is downloaded directly from a public URL when the notebook is executed. This avoids the need to store image files inside the repository and ensures that all learners use the same reference case.

The original image is not square. Directly resizing a rectangular image to (128 \times 128) would distort its anatomical proportions.

We therefore apply the following preprocessing pipeline:

$$
[
\text{Download}
\rightarrow
\text{Grayscale}
\rightarrow
\text{Center Crop}
\rightarrow
\text{Resize}
\rightarrow
\text{Tensor Conversion}
\rightarrow
\text{Normalization}
]
$$

The center crop preserves the middle region of the image while producing equal height and width.

At this stage, the image is kept in the range ([0,1]). It will later be normalized to ([-1,1]) as part of Task 1.



In [ ]:
# Part 1.2 — Download and inspect the reference MRI
# ============================================================

MRI_URL = (
    "https://commons.wikimedia.org/wiki/"
    "Special:Redirect/file/Normal_axial_T2-weighted_MR_image_of_the_brain.jpg"
)


def download_image_from_url(url):
    """
    Download an image from a public URL and return it as a PIL image.

    This function uses Python's built-in urllib instead of requests,
    so no extra package installation is required.
    """

    request = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urlopen(request, timeout=30) as response:
        image_bytes = response.read()

    image = Image.open(BytesIO(image_bytes))

    return image


original_mri = download_image_from_url(MRI_URL)

print("Original image mode:", original_mri.mode)
print("Original image size:", original_mri.size)

plt.figure(figsize=(5, 6))
plt.imshow(original_mri, cmap="gray")
plt.title("Original Axial T2-Weighted Brain MRI")
plt.axis("off")
plt.show()

## Task 1 — Build a Diffusion-Ready MRI Tensor

Your first task is to prepare the downloaded MRI for the diffusion experiment.

Complete the missing parts of the preprocessing pipeline.

The final output must be stored in the variable `x0`.

### Requirements

After preprocessing, `x0` must have:

* shape `[1, 1, 128, 128]`;
* data type `torch.float32`;
* minimum value greater than or equal to `-1`;
* maximum value less than or equal to `1`.

### Intensity normalization

After converting the image into a tensor in the range ([0,1]), map it to ([-1,1]) using:

$$
[
x_{\text{normalized}} = 2x - 1
]
$$

To convert a normalized image back to the display range ([0,1]), use:

$$
[
x_{\text{display}} = \frac{x_{\text{normalized}}+1}{2}
]
$$

The inverse transformation will be used whenever we need to visualize a diffusion tensor as an ordinary image.


In [ ]:
# Task 1 — Prepare the diffusion-ready MRI tensor
# ============================================================
#
# In the original DDPM setting, image values are scaled to [-1, 1]
# before being passed into the diffusion model.

IMAGE_SIZE = 128


def center_square_crop(image):
    """
    Center-crop a PIL image into a square image.

    Parameters
    ----------
    image : PIL.Image
        Input image.

    Returns
    -------
    cropped_image : PIL.Image
        Center-cropped square image.
    """

    width, height = image.size
    crop_size = min(width, height)

    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size

    cropped_image = image.crop(
        (left, top, right, bottom)
    )

    return cropped_image


def pil_to_tensor(image):
    """
    Convert a grayscale PIL image into a PyTorch tensor.

    The output tensor has shape [1, H, W] and values in [0, 1].

    Parameters
    ----------
    image : PIL.Image
        Grayscale input image.

    Returns
    -------
    tensor : torch.Tensor
        Float32 tensor with shape [1, H, W].
    """

    image_array = np.asarray(
        image,
        dtype=np.float32
    )

    image_array = image_array / 255.0

    tensor = torch.from_numpy(
        image_array
    ).float()

    # Add channel dimension:
    # [H, W] -> [1, H, W]
    tensor = tensor.unsqueeze(0)

    return tensor


def normalize_to_neg_one_to_one(image_tensor):
    """
    Convert an image tensor from [0, 1] to [-1, 1].

    Formula:
        x_normalized = 2 * x - 1

    Parameters
    ----------
    image_tensor : torch.Tensor
        Input image tensor in [0, 1].

    Returns
    -------
    normalized_tensor : torch.Tensor
        Normalized image tensor in [-1, 1].
    """

    # WriteYourCodeHere
    normalized_tensor = image_tensor * 2.0 - 1.0

    return normalized_tensor


def denormalize_to_zero_to_one(image_tensor):
    """
    Convert an image tensor from [-1, 1] back to [0, 1].

    Formula:
        x_display = (x_normalized + 1) / 2

    Parameters
    ----------
    image_tensor : torch.Tensor
        Normalized image tensor in [-1, 1].

    Returns
    -------
    display_tensor : torch.Tensor
        Image tensor in [0, 1].
    """

    # WriteYourCodeHere
    display_tensor = (image_tensor + 1.0) / 2.0

    return display_tensor


# ------------------------------------------------------------
# Step 1: Convert the downloaded image to grayscale
# ------------------------------------------------------------

grayscale_mri = original_mri.convert("L")


# ------------------------------------------------------------
# Step 2: Center-crop the image into a square
# ------------------------------------------------------------

cropped_mri = center_square_crop(
    grayscale_mri
)

# ------------------------------------------------------------
# Step 3: Resize the cropped image to 128 x 128
# ------------------------------------------------------------

try:
    resample_method = Image.Resampling.BICUBIC
except AttributeError:
    resample_method = Image.BICUBIC

resized_mri = cropped_mri.resize(
    (IMAGE_SIZE, IMAGE_SIZE),
    resample=resample_method
)

# ------------------------------------------------------------
# Step 4: Convert PIL image to tensor in [0, 1]
# ------------------------------------------------------------

mri_tensor_01 = pil_to_tensor(
    resized_mri
)

# ------------------------------------------------------------
# Step 5: Normalize tensor from [0, 1] to [-1, 1]
# ------------------------------------------------------------

normalized_mri = normalize_to_neg_one_to_one(
    mri_tensor_01
)

# ------------------------------------------------------------
# Step 6: Add batch dimension and move to selected device
# [C, H, W] -> [B, C, H, W]
# ------------------------------------------------------------

x0 = normalized_mri.unsqueeze(0).to(device)


# ------------------------------------------------------------
# Print basic information
# ------------------------------------------------------------

print("Preprocessing completed.")
print("x0 shape:", x0.shape)
print("x0 dtype:", x0.dtype)
print("x0 device:", x0.device)
print("x0 minimum:", x0.min().item())
print("x0 maximum:", x0.max().item())

In [ ]:
# Check Output Cell

assert x0 is not None, (
    "x0 has not been created."
)

assert x0.shape == (1, 1, IMAGE_SIZE, IMAGE_SIZE), (
    f"Expected shape [1, 1, {IMAGE_SIZE}, {IMAGE_SIZE}], "
    f"but received {tuple(x0.shape)}."
)

assert x0.dtype == torch.float32, (
    f"Expected torch.float32, but received {x0.dtype}."
)

assert x0.min().item() >= -1.0001, (
    "The minimum value of x0 is below -1. "
    "Please check the normalization function."
)

assert x0.max().item() <= 1.0001, (
    "The maximum value of x0 is above 1. "
    "Please check the normalization function."
)


# ------------------------------------------------------------
# Verify that normalization and denormalization are inverses
# ------------------------------------------------------------

reconstructed_01 = denormalize_to_zero_to_one(
    x0
)

round_trip_error = torch.max(
    torch.abs(
        reconstructed_01
        -
        mri_tensor_01.unsqueeze(0).to(device)
    )
).item()

assert round_trip_error < 1e-6, (
    "The normalization and denormalization functions "
    "are not exact inverses."
)

print("All preprocessing checks passed.")
print(f"Round-trip error: {round_trip_error:.8f}")


# ------------------------------------------------------------
# Prepare image for display
# ------------------------------------------------------------

display_mri = (
    denormalize_to_zero_to_one(x0)
    .squeeze()
    .detach()
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# Visualize the preprocessing pipeline
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(13, 4)
)

axes[0].imshow(
    grayscale_mri,
    cmap="gray"
)
axes[0].set_title(
    f"Original grayscale MRI\n{grayscale_mri.size}"
)
axes[0].axis("off")


axes[1].imshow(
    resized_mri,
    cmap="gray"
)
axes[1].set_title(
    f"Center-cropped and resized\n{IMAGE_SIZE} x {IMAGE_SIZE}"
)
axes[1].axis("off")


axes[2].imshow(
    display_mri,
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[2].set_title(
    "Diffusion reference image $x_0$"
)
axes[2].axis("off")


plt.tight_layout()
plt.show()

# Part 2 — DDPM Forward Diffusion Process

We have prepared a clean reference MRI image, denoted by $x_0$.

In the DDPM framework, the first step is to define a fixed **forward diffusion process**. This process gradually corrupts the clean image by adding Gaussian noise over many timesteps.

The forward diffusion chain can be written as:

$$
x_0 \rightarrow x_1 \rightarrow x_2 \rightarrow \cdots \rightarrow x_T
$$

where:

* $x_0$ is the clean MRI image;
* $x_t$ is the noisy image at timestep $t$;
* $x_T$ is expected to be close to pure Gaussian noise.

The single-step forward transition is defined as:

$$
q(x_t \mid x_{t-1})
=

\mathcal{N}
\left(
x_t;
\sqrt{1-\beta_t}x_{t-1},
\beta_t I
\right)
$$

Here, $\beta_t$ controls the amount of Gaussian noise added at timestep $t$.

A small value of $\beta_t$ means that only a small amount of noise is added in one step. This makes the reverse denoising process easier to learn, because the model only needs to remove a small amount of corruption at each reverse step.

We define:

$$
\alpha_t = 1 - \beta_t
$$

Using $\alpha_t$, the forward transition can also be written as:

$$
q(x_t \mid x_{t-1})
=

\mathcal{N}
\left(
x_t;
\sqrt{\alpha_t}x_{t-1},
(1-\alpha_t)I
\right)
$$

In this notation:

* $\alpha_t$ represents the amount of signal retained at one step;
* $1-\alpha_t$ represents the amount of noise added at one step;
* repeated application of this process gradually destroys anatomical information.

For the MRI image used in this notebook, the forward process allows us to study how brain structures disappear as the diffusion timestep increases.

For example, we can observe when the ventricles, tissue boundaries, and global brain shape become difficult to recognize.

The next task is to build the DDPM noise schedule, which defines the sequence of $\beta_t$ values used during forward diffusion.


# Part 3 — Building the DDPM Noise Schedule

The forward diffusion process depends on a sequence of variance values:

$$
\beta_1, \beta_2, \ldots, \beta_T
$$

This sequence is called the **noise schedule**.

In the original DDPM setting, the number of diffusion steps is commonly set to:

$$
T = 1000
$$

A linear schedule is used from:

$$
\beta_1 = 10^{-4}
$$

to:

$$
\beta_T = 0.02
$$

This means that early diffusion steps add very little noise, while later diffusion steps add more noise.

After defining $\beta_t$, we compute:

$$
\alpha_t = 1 - \beta_t
$$

Then we compute the cumulative product:

$$
\bar{\alpha}_t
=

\prod_{s=1}^{t}\alpha_s
$$

The value $\bar{\alpha}_t$ is very important because it tells us how much of the original clean image $x_0$ remains after $t$ diffusion steps.

If $\bar{\alpha}_t$ is close to 1, most of the original image signal is still preserved.

If $\bar{\alpha}_t$ is close to 0, most of the original image signal has been destroyed and the image is dominated by Gaussian noise.

Later, we will use $\bar{\alpha}_t$ to sample a noisy image $x_t$ directly from $x_0$ without repeatedly applying all intermediate diffusion steps.


## Task 2 — Build the DDPM Noise Schedule

In the previous section, we introduced the DDPM forward diffusion process. The goal of forward diffusion is to gradually corrupt the clean MRI image $x_0$ by adding Gaussian noise over many timesteps.

In this task, we will build the noise schedule that controls how much noise is added at each timestep.

For our MRI case, imagine that we are preparing a controlled corruption experiment. We start from a clean axial T2-weighted brain MRI and want to observe how its anatomical structures gradually disappear as noise increases.

The clean image is:

$$
x_0
$$

The noisy images are:

$$
x_1, x_2, \ldots, x_T
$$

The noise schedule determines how strongly each step corrupts the image.

---

### What You Need to Do

In this task, you will implement the main variables used by the DDPM forward process:

First, you will create a linear sequence of noise variance values:

$$
\beta_1, \beta_2, \ldots, \beta_T
$$

where $T=1000$.

The first noise value is very small:

$$
\beta_1 = 10^{-4}
$$

and the final noise value is larger:

$$
\beta_T = 0.02
$$

This means that early diffusion steps only slightly perturb the MRI, while later steps add stronger noise.

Next, you will compute:

$$
\alpha_t = 1 - \beta_t
$$

Here, $\alpha_t$ represents how much signal is retained after one diffusion step.

Finally, you will compute the cumulative product:

$$
\bar{\alpha}_t
=

\prod_{s=1}^{t}\alpha_s
$$

The value $\bar{\alpha}_t$ represents how much of the original clean MRI signal remains after $t$ diffusion steps.

---

### Why This Matters

The DDPM forward process will later use $\bar{\alpha}_t$ to sample a noisy image $x_t$ directly from the clean image $x_0$:

$$
x_t
=

\sqrt{\bar{\alpha}_t}x_0
+
\sqrt{1-\bar{\alpha}_t}\epsilon
$$

where:

$$
\epsilon \sim \mathcal{N}(0,I)
$$

This equation shows that $x_t$ is a mixture of two parts:

* the remaining clean MRI signal, controlled by $\sqrt{\bar{\alpha}_t}$;
* the added Gaussian noise, controlled by $\sqrt{1-\bar{\alpha}_t}$.

Therefore, building the noise schedule is necessary before we can visualize how the MRI changes at different diffusion timesteps.

---

### Expected Results

After completing this task, you should obtain the following tensors:

```text
betas
alphas
alpha_bars
sqrt_alpha_bars
sqrt_one_minus_alpha_bars
```

Their expected shapes are:

```text
torch.Size([1000])
```

You should also observe the following behavior:

The values of $\beta_t$ should increase linearly from $10^{-4}$ to $0.02$.

The values of $\bar{\alpha}_t$ should gradually decrease as timestep increases.

The clean-image signal coefficient $\sqrt{\bar{\alpha}_t}$ should decrease over time.

The noise coefficient $\sqrt{1-\bar{\alpha}_t}$ should increase over time.

This means that as $t$ becomes larger, less anatomical information remains from the original MRI, and the image becomes increasingly dominated by Gaussian noise.


In [ ]:
# ============================================================
# Task 2 — Build the DDPM noise schedule


# ------------------------------------------------------------
# Step 0:
# Define the basic DDPM noise schedule parameters.
#
# The original DDPM paper commonly uses T = 1000 diffusion steps.
# beta_start is very small, so early steps add only weak noise.
# beta_end is larger, so later steps add stronger noise.
# ------------------------------------------------------------

T = 1000

beta_start = 1e-4
beta_end = 2e-2


# ------------------------------------------------------------
# Step 1:
# Create a function for the linear beta schedule.
#
# A linear schedule means that beta_t increases evenly from
# beta_start to beta_end across all timesteps.
#
# torch.linspace(start, end, steps) creates evenly spaced values.
# ------------------------------------------------------------

def make_linear_beta_schedule(
    timesteps,
    beta_start,
    beta_end
):
    """
    Create a linear beta schedule for DDPM.

    Parameters
    ----------
    timesteps : int
        Total number of diffusion timesteps.

    beta_start : float
        The first beta value.

    beta_end : float
        The final beta value.

    Returns
    -------
    betas : torch.Tensor
        A tensor containing beta_1, beta_2, ..., beta_T.
    """

    # WriteYourCodeHere
    betas = torch.linspace(
        beta_start,
        beta_end,
        timesteps,
        dtype=torch.float32
    )

    return betas


# ------------------------------------------------------------
# Step 2:
# Build the beta schedule.
#
# After this step, betas[t] stores the amount of noise added
# at timestep t.
#
# Note:
# Python indexing starts from 0, so:
# betas[0] corresponds to beta_1
# betas[T - 1] corresponds to beta_T
# ------------------------------------------------------------

# WriteYourCodeHere
betas = make_linear_beta_schedule(
    timesteps=T,
    beta_start=beta_start,
    beta_end=beta_end
).to(device)


# ------------------------------------------------------------
# Step 3:
# Compute alpha_t.
#
# In DDPM, beta_t represents the noise added at one step.
# Therefore, alpha_t = 1 - beta_t represents the signal retained
# after that step.
# ------------------------------------------------------------

# WriteYourCodeHere
alphas = 1.0 - betas


# ------------------------------------------------------------
# Step 4:
# Compute alpha_bar_t.
#
# alpha_bar_t is the cumulative product of alpha values:
#
# alpha_bar_t = alpha_1 * alpha_2 * ... * alpha_t
#
# It measures how much of the original clean image x0 remains
# after t diffusion steps.
# ------------------------------------------------------------

# WriteYourCodeHere
alpha_bars = torch.cumprod(
    alphas,
    dim=0
)


# ------------------------------------------------------------
# Step 5:
# Compute the signal and noise coefficients used later.
#
# In the closed-form DDPM forward process:
#
# x_t = sqrt(alpha_bar_t) * x_0
#       + sqrt(1 - alpha_bar_t) * epsilon
#
# sqrt(alpha_bar_t) controls the clean image signal.
# sqrt(1 - alpha_bar_t) controls the Gaussian noise.
# ------------------------------------------------------------

# WriteYourCodeHere
sqrt_alpha_bars = torch.sqrt(
    alpha_bars
)

# WriteYourCodeHere
sqrt_one_minus_alpha_bars = torch.sqrt(
    1.0 - alpha_bars
)


# ------------------------------------------------------------
# Checkpoint:
# These checks verify that the schedule has been implemented
# correctly.
#
# They are not part of the DDPM mathematical definition.
# They are used here to catch common coding mistakes.
# ------------------------------------------------------------

assert betas.shape == (T,), (
    f"Expected betas shape ({T},), but got {betas.shape}."
)

assert alphas.shape == (T,), (
    f"Expected alphas shape ({T},), but got {alphas.shape}."
)

assert alpha_bars.shape == (T,), (
    f"Expected alpha_bars shape ({T},), but got {alpha_bars.shape}."
)

assert torch.all(betas > 0), (
    "All beta values should be positive."
)

assert torch.all(betas < 1), (
    "All beta values should be smaller than 1."
)

assert torch.all(alphas > 0), (
    "All alpha values should be positive."
)

assert torch.all(alphas <= 1), (
    "All alpha values should be less than or equal to 1."
)

assert torch.all(alpha_bars[1:] <= alpha_bars[:-1]), (
    "alpha_bars should decrease as timestep increases."
)


# ------------------------------------------------------------
# Step 6:
# Print important values.
#
# These values help us confirm the behavior of the schedule.
# beta_1 should be close to 1e-4.
# beta_T should be close to 0.02.
# alpha_bar_T should be close to 0, meaning most original signal
# has been removed by the final timestep.
# ------------------------------------------------------------

print("DDPM noise schedule created successfully.")
print()
print("Number of timesteps T:", T)
print("betas shape:", betas.shape)
print("alphas shape:", alphas.shape)
print("alpha_bars shape:", alpha_bars.shape)
print()
print("beta_1:", betas[0].item())
print("beta_T:", betas[-1].item())
print("alpha_1:", alphas[0].item())
print("alpha_T:", alphas[-1].item())
print("alpha_bar_1:", alpha_bars[0].item())
print("alpha_bar_T:", alpha_bars[-1].item())


# ------------------------------------------------------------
# Step 7:
# Visualize the noise schedule.
#
# The first plot shows beta_t.
# The second plot shows alpha_bar_t.
# The third plot compares the clean-image signal coefficient
# and the Gaussian-noise coefficient.
# ------------------------------------------------------------

timesteps = torch.arange(
    T,
    device=device
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4)
)


# Plot 1: beta schedule
axes[0].plot(
    timesteps.cpu().numpy(),
    betas.detach().cpu().numpy()
)
axes[0].set_title(r"Linear noise schedule $\beta_t$")
axes[0].set_xlabel("Timestep")
axes[0].set_ylabel(r"$\beta_t$")


# Plot 2: cumulative signal retention
axes[1].plot(
    timesteps.cpu().numpy(),
    alpha_bars.detach().cpu().numpy()
)
axes[1].set_title(r"Cumulative signal retention $\bar{\alpha}_t$")
axes[1].set_xlabel("Timestep")
axes[1].set_ylabel(r"$\bar{\alpha}_t$")


# Plot 3: signal coefficient versus noise coefficient
axes[2].plot(
    timesteps.cpu().numpy(),
    sqrt_alpha_bars.detach().cpu().numpy(),
    label=r"Signal coefficient $\sqrt{\bar{\alpha}_t}$"
)

axes[2].plot(
    timesteps.cpu().numpy(),
    sqrt_one_minus_alpha_bars.detach().cpu().numpy(),
    label=r"Noise coefficient $\sqrt{1-\bar{\alpha}_t}$"
)

axes[2].set_title("Signal and noise coefficients")
axes[2].set_xlabel("Timestep")
axes[2].set_ylabel("Coefficient")
axes[2].legend()


plt.tight_layout()
plt.show()

# Part 4 — DDPM Forward Sampling: Visualizing MRI Corruption

In the previous part, we built the DDPM noise schedule.

We now have the cumulative signal retention coefficient $\bar{\alpha}_t$, which tells us how much of the original clean MRI signal remains after $t$ diffusion steps.

The next question is:

How do we actually generate a noisy MRI image $x_t$ at a specific timestep?

A direct way would be to repeatedly apply the one-step transition:

$$
x_0 \rightarrow x_1 \rightarrow x_2 \rightarrow \cdots \rightarrow x_t
$$

However, DDPM provides a closed-form expression that allows us to sample $x_t$ directly from $x_0$:

$$
q(x_t \mid x_0)
=

\mathcal{N}
\left(
x_t;
\sqrt{\bar{\alpha}_t}x_0,
(1-\bar{\alpha}_t)I
\right)
$$

Equivalently, we can write:

$$
x_t
=

\sqrt{\bar{\alpha}_t}x_0
+
\sqrt{1-\bar{\alpha}_t}\epsilon
$$

where:

$$
\epsilon \sim \mathcal{N}(0,I)
$$

This equation shows that a noisy image $x_t$ is a weighted combination of two components:

* the remaining clean MRI signal $\sqrt{\bar{\alpha}_t}x_0$;
* the added Gaussian noise $\sqrt{1-\bar{\alpha}_t}\epsilon$.

When $t$ is small, $\bar{\alpha}_t$ is close to 1. Therefore, the clean MRI signal dominates.

When $t$ is large, $\bar{\alpha}_t$ becomes close to 0. Therefore, Gaussian noise dominates.

In this part, we will use this equation to visualize how the axial T2-weighted brain MRI gradually loses anatomical structure as the diffusion timestep increases.


## Task 3 — Sample Noisy MRI Images at Different Timesteps

In this task, we will implement the closed-form DDPM forward sampling equation.

The clean input image is the preprocessed MRI tensor:

$$
x_0
$$

For a selected timestep $t$, we want to sample a noisy image:

$$
x_t
$$

using:

$$
x_t
===

\sqrt{\bar{\alpha}_t}x_0
+
\sqrt{1-\bar{\alpha}_t}\epsilon
$$

where $\epsilon$ is randomly sampled Gaussian noise.

---

### What You Need to Do

You will implement two key helper functions.

First, you will implement `extract()`.

This function takes a one-dimensional schedule tensor, such as `sqrt_alpha_bars`, and extracts the values corresponding to the selected timestep.

This is necessary because the image tensor has shape:

$$
[B,C,H,W]
$$

but the schedule tensor has shape:

$$
[T]
$$

Therefore, we need to reshape the extracted coefficient so it can be broadcast across the image dimensions.

Second, you will implement `q_sample()`.

This function applies the closed-form DDPM forward process and returns:

$$
x_t
$$

for any selected timestep.



### MRI Experiment

In this task, we will sample noisy MRI images at multiple timesteps.

For example:

$$
t = 0, 50, 100, 200, 400, 600, 800, 999
$$

At early timesteps, the ventricles, tissue boundaries, and global brain shape should still be visible.

At later timesteps, these anatomical structures should gradually disappear.

By the final timestep, the image should be dominated by Gaussian noise.

---

### Expected Results

After completing this task, you should see:

* the original clean MRI image $x_0$;
* several noisy MRI images $x_t$ at different timesteps;
* a plot showing how the signal-to-noise ratio changes over time.

The expected trend is:

* as $t$ increases, $\sqrt{\bar{\alpha}_t}$ decreases;
* as $t$ increases, $\sqrt{1-\bar{\alpha}_t}$ increases;
* as $t$ increases, the MRI becomes less anatomically recognizable.


In [ ]:
# ============================================================
# Task 3 — Sample noisy MRI images at different timesteps


# ------------------------------------------------------------
# Step 0:
# Define a helper function to extract timestep-specific values.
#
# Why do we need this?
#
# The schedule tensors have shape [T].
# For example:
# sqrt_alpha_bars.shape = [1000]
#
# The image tensor x0 has shape [B, C, H, W].
# For example:
# x0.shape = [1, 1, 128, 128]
#
# When we choose a timestep t, we need to extract the scalar
# coefficient at that timestep and reshape it so that PyTorch
# can broadcast it across the full image tensor.
# ------------------------------------------------------------

def extract(schedule_values, timesteps, target_shape):
    """
    Extract values from a 1-D diffusion schedule at selected timesteps.

    Parameters
    ----------
    schedule_values : torch.Tensor
        A 1-D tensor with shape [T].
        Example: sqrt_alpha_bars.

    timesteps : torch.Tensor
        A tensor with shape [B], containing timestep indices.

    target_shape : torch.Size
        The shape of the target image tensor, usually [B, C, H, W].

    Returns
    -------
    extracted_values : torch.Tensor
        A tensor with shape [B, 1, 1, 1], ready for broadcasting.
    """

    batch_size = timesteps.shape[0]

    # WriteYourCodeHere
    extracted_values = schedule_values.gather(
        dim=0,
        index=timesteps
    )

    # WriteYourCodeHere
    extracted_values = extracted_values.reshape(
        batch_size,
        *((1,) * (len(target_shape) - 1))
    )

    return extracted_values


# ------------------------------------------------------------
# Step 1:
# Implement the closed-form DDPM forward sampling function.
#
# q_sample takes a clean image x_start and returns a noisy image x_t.
#
# The random noise epsilon has the same shape as the image.
# If no noise is provided, we generate standard Gaussian noise.
# ------------------------------------------------------------

def q_sample(x_start, timesteps, noise=None):
    """
    Sample x_t from x_0 using the closed-form DDPM forward process.

    Parameters
    ----------
    x_start : torch.Tensor
        Clean image tensor x0 with shape [B, C, H, W].

    timesteps : torch.Tensor
        Timestep tensor with shape [B].

    noise : torch.Tensor or None
        Gaussian noise epsilon.
        If None, random Gaussian noise will be generated.

    Returns
    -------
    x_noisy : torch.Tensor
        Noisy image tensor x_t.

    noise : torch.Tensor
        The Gaussian noise epsilon used to create x_t.
    """

    if noise is None:
        # WriteYourCodeHere
        noise = torch.randn_like(
            x_start
        )

    # WriteYourCodeHere
    signal_coefficient = extract(
        sqrt_alpha_bars,
        timesteps,
        x_start.shape
    )

    # WriteYourCodeHere
    noise_coefficient = extract(
        sqrt_one_minus_alpha_bars,
        timesteps,
        x_start.shape
    )

    # WriteYourCodeHere
    x_noisy = (
        signal_coefficient * x_start
        +
        noise_coefficient * noise
    )

    return x_noisy, noise


# ------------------------------------------------------------
# Checkpoint:
# These checks help verify that q_sample is implemented correctly.
# They are used to catch common shape and range mistakes.
# ------------------------------------------------------------

test_timesteps = torch.tensor(
    [0],
    device=device,
    dtype=torch.long
)

test_noisy, test_noise = q_sample(
    x_start=x0,
    timesteps=test_timesteps
)

assert test_noisy.shape == x0.shape, (
    f"Expected noisy image shape {x0.shape}, "
    f"but received {test_noisy.shape}."
)

assert test_noise.shape == x0.shape, (
    f"Expected noise shape {x0.shape}, "
    f"but received {test_noise.shape}."
)

assert torch.isfinite(test_noisy).all(), (
    "The sampled noisy image contains non-finite values."
)

assert torch.isfinite(test_noise).all(), (
    "The sampled noise contains non-finite values."
)


# ------------------------------------------------------------
# Step 2:
# Select several timesteps for visual inspection.
#
# Python uses zero-based indexing.
# Therefore, timestep index 0 corresponds to the first diffusion step.
# timestep index 999 corresponds to the final step when T = 1000.
# ------------------------------------------------------------

selected_timesteps = [
    0,
    50,
    100,
    200,
    400,
    600,
    800,
    999
]


# ------------------------------------------------------------
# Step 3:
# Use the same random noise pattern for all selected timesteps.
#
# Why?
#
# If each timestep used a different random noise tensor, then changes
# in the images would come from both the timestep and the random noise.
#
# By reusing the same epsilon, we make the comparison cleaner:
# the only changing factor is the diffusion timestep.
# ------------------------------------------------------------

# WriteYourCodeHere
shared_noise = torch.randn_like(
    x0
)


noisy_images = []

for timestep in selected_timesteps:

    t_tensor = torch.tensor(
        [timestep],
        device=device,
        dtype=torch.long
    )

    x_t, _ = q_sample(
        x_start=x0,
        timesteps=t_tensor,
        noise=shared_noise
    )

    noisy_images.append(
        x_t
    )


# ------------------------------------------------------------
# Step 4:
# Print the signal and noise coefficients for selected timesteps.
#
# This helps connect the visualization back to the formula:
#
# x_t = sqrt(alpha_bar_t) * x0
#       + sqrt(1 - alpha_bar_t) * epsilon
# ------------------------------------------------------------

print("Selected timestep coefficients")
print("--------------------------------")

for timestep in selected_timesteps:
    signal_value = sqrt_alpha_bars[timestep].item()
    noise_value = sqrt_one_minus_alpha_bars[timestep].item()

    print(
        f"t = {timestep:3d} | "
        f"signal coefficient = {signal_value:.4f} | "
        f"noise coefficient = {noise_value:.4f}"
    )


# ------------------------------------------------------------
# Step 5:
# Visualize the clean MRI and noisy MRI images.
#
# The model operates on images normalized to [-1, 1].
# For visualization, we convert images back to [0, 1].
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    len(selected_timesteps) + 1,
    figsize=(18, 3)
)


clean_display = (
    denormalize_to_zero_to_one(x0)
    .squeeze()
    .detach()
    .cpu()
    .numpy()
)

axes[0].imshow(
    clean_display,
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[0].set_title("Clean\n$x_0$")
axes[0].axis("off")


for image_index, timestep in enumerate(selected_timesteps):

    noisy_display = (
        denormalize_to_zero_to_one(noisy_images[image_index])
        .clamp(0, 1)
        .squeeze()
        .detach()
        .cpu()
        .numpy()
    )

    axes[image_index + 1].imshow(
        noisy_display,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[image_index + 1].set_title(
        f"t = {timestep}"
    )

    axes[image_index + 1].axis("off")


plt.suptitle(
    "DDPM Forward Diffusion: MRI Corruption over Timesteps",
    y=1.05
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Step 6:
# Analyze the signal-to-noise ratio over time.
#
# The theoretical signal-to-noise ratio can be computed as:
#
# SNR_t = alpha_bar_t / (1 - alpha_bar_t)
#
# When SNR is high, the image signal dominates.
# When SNR is low, Gaussian noise dominates.
# ------------------------------------------------------------

# WriteYourCodeHere
snr = alpha_bars / (
    1.0 - alpha_bars
)

# WriteYourCodeHere
snr_db = 10.0 * torch.log10(
    snr
)


timesteps = torch.arange(
    T,
    device=device
)


plt.figure(figsize=(7, 4))

plt.plot(
    timesteps.detach().cpu().numpy(),
    snr_db.detach().cpu().numpy()
)

plt.title("Signal-to-Noise Ratio during DDPM Forward Diffusion")
plt.xlabel("Timestep")
plt.ylabel("SNR (dB)")
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Step 7:
# Final interpretation.
# ------------------------------------------------------------

print()
print("Task 3 completed.")
print("As timestep increases, the clean MRI signal decreases and Gaussian noise dominates.")
print("This confirms the role of the DDPM forward process as a controlled corruption process.")

# Part 5 — DDPM Noise Prediction: Recovering the Clean MRI

In the previous part, we used the DDPM forward process to generate noisy MRI images.

For a clean image $x_0$, the forward process samples a noisy image $x_t$ using:

$$x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon$$

where $\epsilon \sim \mathcal{N}(0,I)$.

This equation tells us that $x_t$ is created from two components: the remaining clean image signal and the added Gaussian noise.

The key idea of DDPM training is to learn a neural network that predicts the noise $\epsilon$ from the noisy image $x_t$ and the timestep $t$.

The neural network is usually written as:

$$\epsilon_\theta(x_t,t)$$

Here, $\theta$ represents the learnable parameters of the neural network.

The simplified DDPM training objective is:

$$\mathcal{L}*{\mathrm{simple}} = \mathbb{E}*{x_0,t,\epsilon}\left[\left|\epsilon - \epsilon_\theta(x_t,t)\right|_2^2\right]$$

This means that the model is trained to make its predicted noise $\epsilon_\theta(x_t,t)$ as close as possible to the true noise $\epsilon$ used in the forward process.

Once the noise has been predicted, we can rearrange the forward equation to estimate the original clean image:

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}*t}\epsilon*\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}}$$

This equation is important because it shows why noise prediction is useful.

If the predicted noise is accurate, then the estimated clean image $\hat{x}_0$ should be close to the original clean MRI $x_0$.

If the predicted noise is inaccurate, then the reconstructed image will contain errors, artifacts, or distorted anatomical structures.

In this notebook, we do not train a full U-Net diffusion model. Instead, we use a controlled experiment to understand the mechanism.

We will compare three cases:

* using the true noise $\epsilon$ as an oracle prediction;
* using a slightly imperfect noise prediction;
* using a more inaccurate noise prediction.

This allows us to directly observe how noise-prediction accuracy affects MRI reconstruction quality.


## Task 4 — Estimate the Clean MRI from Predicted Noise

In this task, we will study how DDPM recovers an estimate of the clean image from a noisy image.

We start with the clean MRI tensor:

$$x_0$$

Then we choose one diffusion timestep $t$ and generate a noisy MRI image:

$$x_t$$

using the forward sampling function from Part 4.

The key reconstruction equation is:

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}*t}\epsilon*\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}}$$

---

### What You Need to Do

You will implement a function called `predict_x0_from_noise()`.

This function takes three inputs:

* the noisy image $x_t$;
* the timestep $t$;
* the predicted noise $\epsilon_\theta(x_t,t)$.

It returns the estimated clean image: {x}_0


### MRI Experiment

In a real DDPM model, the predicted noise would come from a trained neural network.

In this educational experiment, we simulate the prediction process.

First, we use the true noise as an oracle prediction. This represents an ideal model that predicts the noise perfectly.

Then, we add different levels of random error to the true noise. This represents imperfect models with different prediction qualities.

We will compare:

* the original clean MRI;
* the noisy MRI at timestep $t$;
* the reconstructed MRI using oracle noise;
* the reconstructed MRI using imperfect noise predictions.

---

### Expected Results

If the predicted noise is exactly correct, the reconstructed image $\hat{x}_0$ should be almost identical to the original MRI $x_0$.

As the predicted noise becomes less accurate, the reconstruction error should increase.

In the visualization, anatomical details should be clearest in the oracle reconstruction and become less reliable as the prediction error increases.


In [ ]:
# ============================================================
# Task 4 — Estimate the clean MRI from predicted noise

# ------------------------------------------------------------
# Step 0:
# Select one diffusion timestep for the reconstruction experiment.
#
# We choose a middle-to-late timestep so that the noisy MRI is
# visibly corrupted, but still numerically stable for reconstruction.
# ------------------------------------------------------------

target_timestep = 500

t_tensor = torch.tensor(
    [target_timestep],
    device=device,
    dtype=torch.long
)


# ------------------------------------------------------------
# Step 1:
# Generate one noisy MRI image x_t.
#
# We explicitly create the true Gaussian noise epsilon here.
# This allows us to later compare oracle noise prediction with
# imperfect noise prediction.
# ------------------------------------------------------------

# WriteYourCodeHere
true_noise = torch.randn_like(
    x0
)

# WriteYourCodeHere
x_t, true_noise = q_sample(
    x_start=x0,
    timesteps=t_tensor,
    noise=true_noise
)


# ------------------------------------------------------------
# Step 2:
# Implement the clean-image estimation function.
#
# This function rearranges the DDPM forward equation:
#
# x_t = sqrt(alpha_bar_t) * x0
#       + sqrt(1 - alpha_bar_t) * epsilon
#
# to solve for x0.
# ------------------------------------------------------------

def predict_x0_from_noise(
    x_noisy,
    timesteps,
    predicted_noise
):
    """
    Estimate the clean image x0 from a noisy image x_t
    and a predicted noise tensor.

    Parameters
    ----------
    x_noisy : torch.Tensor
        The noisy image x_t with shape [B, C, H, W].

    timesteps : torch.Tensor
        The selected timestep tensor with shape [B].

    predicted_noise : torch.Tensor
        The predicted Gaussian noise epsilon_theta(x_t, t).

    Returns
    -------
    x0_hat : torch.Tensor
        The estimated clean image.
    """

    # Step 2.1:
    # Extract sqrt(alpha_bar_t), which tells us how much clean signal
    # remains at the selected timestep.
    # WriteYourCodeHere
    signal_coefficient = extract(
        sqrt_alpha_bars,
        timesteps,
        x_noisy.shape
    )

    # Step 2.2:
    # Extract sqrt(1 - alpha_bar_t), which tells us how strongly
    # the Gaussian noise contributes to x_t.
    # WriteYourCodeHere
    noise_coefficient = extract(
        sqrt_one_minus_alpha_bars,
        timesteps,
        x_noisy.shape
    )

    # Step 2.3:
    # Rearrange the forward equation to estimate x0.
    # WriteYourCodeHere
    x0_hat = (
        x_noisy
        -
        noise_coefficient * predicted_noise
    ) / signal_coefficient

    return x0_hat


# ------------------------------------------------------------
# Step 3:
# Case 1 — Oracle noise prediction.
#
# In this ideal case, the predicted noise is exactly the same
# as the true noise used to create x_t.
#
# A perfect noise prediction should reconstruct x0 almost exactly.
# ------------------------------------------------------------

# WriteYourCodeHere
oracle_predicted_noise = true_noise

# WriteYourCodeHere
x0_hat_oracle = predict_x0_from_noise(
    x_noisy=x_t,
    timesteps=t_tensor,
    predicted_noise=oracle_predicted_noise
)


# ------------------------------------------------------------
# Step 4:
# Case 2 — Imperfect noise prediction.
#
# A real neural network will not predict the noise perfectly.
# To simulate this, we add random prediction errors to the true noise.
#
# Larger error_scale means a less accurate noise predictor.
# ------------------------------------------------------------

error_scales = [
    0.05,
    0.10,
    0.20
]

reconstruction_results = [
    {
        "label": "Oracle",
        "image": x0_hat_oracle,
        "error_scale": 0.00
    }
]

for error_scale in error_scales:

    # Step 4.1:
    # Create a random prediction error.
    # WriteYourCodeHere
    prediction_error = error_scale * torch.randn_like(
        true_noise
    )

    # Step 4.2:
    # Simulate an imperfect neural network prediction.
    # WriteYourCodeHere
    imperfect_predicted_noise = true_noise + prediction_error

    # Step 4.3:
    # Estimate x0 using the imperfect predicted noise.
    # WriteYourCodeHere
    x0_hat_imperfect = predict_x0_from_noise(
        x_noisy=x_t,
        timesteps=t_tensor,
        predicted_noise=imperfect_predicted_noise
    )

    reconstruction_results.append(
        {
            "label": f"Error scale {error_scale}",
            "image": x0_hat_imperfect,
            "error_scale": error_scale
        }
    )


# ------------------------------------------------------------
# Step 5:
# Checkpoint.
#
# These checks help confirm that the reconstruction tensors have
# the correct shape and contain valid numerical values.
# ------------------------------------------------------------

assert x_t.shape == x0.shape, (
    f"Expected x_t shape {x0.shape}, but got {x_t.shape}."
)

assert x0_hat_oracle.shape == x0.shape, (
    f"Expected x0_hat_oracle shape {x0.shape}, "
    f"but got {x0_hat_oracle.shape}."
)

assert torch.isfinite(x0_hat_oracle).all(), (
    "Oracle reconstruction contains non-finite values."
)

oracle_mse = torch.mean(
    (x0_hat_oracle - x0) ** 2
).item()

assert oracle_mse < 1e-8, (
    "Oracle reconstruction should be almost exact. "
    "Please check predict_x0_from_noise()."
)


# ------------------------------------------------------------
# Step 6:
# Compute reconstruction metrics.
#
# Mean squared error measures average squared pixel error.
# Mean absolute error measures average absolute pixel error.
# Lower values mean better reconstruction.
# ------------------------------------------------------------

metrics = []

for result in reconstruction_results:

    reconstructed_image = result["image"]

    mse = torch.mean(
        (reconstructed_image - x0) ** 2
    ).item()

    mae = torch.mean(
        torch.abs(reconstructed_image - x0)
    ).item()

    metrics.append(
        {
            "label": result["label"],
            "error_scale": result["error_scale"],
            "mse": mse,
            "mae": mae
        }
    )


# ------------------------------------------------------------
# Step 7:
# Print timestep information and reconstruction metrics.
# ------------------------------------------------------------

signal_value = sqrt_alpha_bars[target_timestep].item()
noise_value = sqrt_one_minus_alpha_bars[target_timestep].item()

print("DDPM clean-image estimation experiment")
print("--------------------------------------")
print(f"Selected timestep: t = {target_timestep}")
print(f"Signal coefficient sqrt(alpha_bar_t): {signal_value:.4f}")
print(f"Noise coefficient sqrt(1 - alpha_bar_t): {noise_value:.4f}")
print()
print("Reconstruction metrics")
print("--------------------------------------")

for item in metrics:
    print(
        f"{item['label']:>16s} | "
        f"MSE = {item['mse']:.8f} | "
        f"MAE = {item['mae']:.8f}"
    )


# ------------------------------------------------------------
# Step 8:
# Visualize the original, noisy, and reconstructed MRI images.
#
# The diffusion tensors are stored in [-1, 1].
# For display, we convert them back to [0, 1] and clamp values
# to avoid visualization artifacts.
# ------------------------------------------------------------

images_to_show = [
    {
        "title": "Clean MRI\n$x_0$",
        "image": x0
    },
    {
        "title": f"Noisy MRI\n$x_t$, t={target_timestep}",
        "image": x_t
    }
]

for result in reconstruction_results:
    images_to_show.append(
        {
            "title": result["label"],
            "image": result["image"]
        }
    )


fig, axes = plt.subplots(
    1,
    len(images_to_show),
    figsize=(18, 3.5)
)

for axis, item in zip(axes, images_to_show):

    display_image = (
        denormalize_to_zero_to_one(item["image"])
        .clamp(0, 1)
        .squeeze()
        .detach()
        .cpu()
        .numpy()
    )

    axis.imshow(
        display_image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axis.set_title(
        item["title"]
    )

    axis.axis("off")


plt.suptitle(
    "DDPM Clean-Image Estimation from Predicted Noise",
    y=1.05
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Step 9:
# Plot reconstruction error as prediction error increases.
#
# This plot shows that inaccurate noise prediction leads to
# worse clean-image reconstruction.
# ------------------------------------------------------------

error_scale_values = [
    item["error_scale"] for item in metrics
]

mse_values = [
    item["mse"] for item in metrics
]

mae_values = [
    item["mae"] for item in metrics
]


plt.figure(figsize=(7, 4))

plt.plot(
    error_scale_values,
    mse_values,
    marker="o",
    label="MSE"
)

plt.plot(
    error_scale_values,
    mae_values,
    marker="o",
    label="MAE"
)

plt.title("Reconstruction Error versus Noise Prediction Error")
plt.xlabel("Prediction error scale")
plt.ylabel("Reconstruction error")
plt.legend()
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Step 10:
# Final interpretation.
# ------------------------------------------------------------

print()
print("Task 4 completed.")
print("The oracle prediction reconstructs the clean MRI almost exactly.")
print("As prediction error increases, reconstruction quality decreases.")
print("This demonstrates why accurate noise prediction is central to DDPM.")

# Part 6 — DDIM Fast Sampling: Reducing the Number of Reverse Steps

DDPM sampling is usually slow because the reverse process follows many sequential denoising steps.

If the original diffusion process uses $T=1000$, then standard DDPM sampling may require approximately 1000 reverse model evaluations.

DDIM provides a faster alternative.

The key idea is that we do not always need to visit every timestep. Instead, we can choose a smaller subset of timesteps:

$$\tau = {\tau_1,\tau_2,\ldots,\tau_S}$$

where $S \ll T$.

For example, instead of using 1000 reverse steps, we may use 100, 50, or 20 steps.

DDIM uses the same kind of noise prediction model:

$$\epsilon_\theta(x_t,t)$$

but applies a different deterministic update rule when $\eta=0$.

First, we estimate the clean image from the current noisy image:

$$\hat{x}_0 = \frac{x_t - \sqrt{1-\bar{\alpha}*t}\epsilon*\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}}$$

Then, we move from the current timestep $t$ to a previous timestep $t_{\mathrm{prev}}$ using:

$$x_{t_{\mathrm{prev}}} = \sqrt{\bar{\alpha}*{t*{\mathrm{prev}}}}\hat{x}*0 + \sqrt{1-\bar{\alpha}*{t_{\mathrm{prev}}}}\epsilon_\theta(x_t,t)$$

This is the deterministic DDIM update.

When $t_{\mathrm{prev}}=-1$, we treat $\bar{\alpha}*{t*{\mathrm{prev}}}=1$, so the final result becomes the estimated clean image.

In this notebook, we do not train a full medical diffusion model. Instead, we use an oracle-assisted controlled reconstruction experiment. This allows us to focus on the DDIM update rule and timestep reduction mechanism without claiming that we have trained a clinically valid generative model.


## Task 5 — Implement Reduced-Step DDIM Reconstruction

In this task, we will implement deterministic DDIM reconstruction using fewer reverse steps.

We start from a highly noisy MRI image:

$$x_T$$

Then we reconstruct it back toward the clean image by following a selected sequence of reverse timesteps.

---

### What You Need to Do

You will implement three components.

First, you will create a function called `make_ddim_timesteps()`.

This function selects a smaller subset of timesteps from the original DDPM schedule.

For example, if the original schedule has $T=1000$, we may choose only:

$$S=100,\quad S=50,\quad S=20$$

reverse steps.

Second, you will implement `ddim_step()`.

This function performs one deterministic DDIM update from the current timestep $t$ to a previous timestep $t_{\mathrm{prev}}$.

Third, you will run a controlled reconstruction experiment using oracle noise prediction.

In a real model, $\epsilon_\theta(x_t,t)$ would be predicted by a trained neural network. In this simplified experiment, we use the true noise as an oracle prediction so that we can isolate and understand the DDIM update mechanism.


### MRI Experiment

We will generate a noisy MRI at the final diffusion timestep.

Then we will reconstruct it using different numbers of DDIM steps:

```text
1000 steps
100 steps
50 steps
20 steps
10 steps
```

The number of steps represents the number of reverse denoising updates.

A smaller number of steps means fewer model evaluations and faster sampling.

---

### Expected Results

Because this is an oracle-assisted experiment, the reconstruction can remain very accurate even with fewer steps.

This is different from a real trained diffusion model, where fewer steps may reduce image quality.

The main expected result in this task is to observe that DDIM can define a valid reduced-step reverse trajectory.

You should see:

* the highly noisy MRI image $x_T$;
* reconstructed MRI images using different numbers of DDIM steps;
* a comparison of step counts and reconstruction errors.

This task demonstrates the mechanism behind DDIM acceleration, not clinical image generation.


In [ ]:
# ============================================================
# Task 5 — Implement reduced-step DDIM reconstruction


# ------------------------------------------------------------
# Step 0:
# Check that required variables and functions from previous parts
# are already available.
#
# This task depends on:
# x0, T, alpha_bars, q_sample, predict_x0_from_noise,
# denormalize_to_zero_to_one, and device.
# ------------------------------------------------------------

required_names = [
    "x0",
    "T",
    "alpha_bars",
    "q_sample",
    "predict_x0_from_noise",
    "denormalize_to_zero_to_one",
    "device"
]

for name in required_names:
    assert name in globals(), (
        f"Required object '{name}' is missing. "
        "Please run the previous parts before running Task 5."
    )


# ------------------------------------------------------------
# Step 1:
# Create a helper function for retrieving alpha_bar values.
#
# Why do we need this?
#
# For normal timesteps, we read alpha_bar_t from the DDPM schedule.
#
# For the final clean-image state, we use previous_timestep = -1.
# In that special case, alpha_bar is treated as 1.
#
# This means:
# sqrt(alpha_bar_prev) = 1
# sqrt(1 - alpha_bar_prev) = 0
#
# Therefore, the final update returns the clean estimate.
# ------------------------------------------------------------

def get_alpha_bar_at_timestep(timestep):
    """
    Return alpha_bar at a given timestep.

    Parameters
    ----------
    timestep : int
        Diffusion timestep index.
        If timestep == -1, return alpha_bar = 1.

    Returns
    -------
    alpha_bar_value : torch.Tensor
        Scalar tensor on the selected device.
    """

    if timestep < 0:
        alpha_bar_value = torch.tensor(
            1.0,
            device=device,
            dtype=torch.float32
        )
    else:
        alpha_bar_value = alpha_bars[timestep]

    return alpha_bar_value


# ------------------------------------------------------------
# Step 2:
# Select a reduced DDIM timestep sequence.
#
# The original DDPM schedule has T = 1000 timesteps.
# DDIM can use only a subset of these timesteps.
#
# For example:
# num_steps = 50 means that we select around 50 timesteps
# between 0 and T - 1.
#
# Reverse sampling goes from noisy to clean:
#
# T - 1 -> ... -> 0 -> -1
#
# where -1 represents the final clean-image estimate.
# ------------------------------------------------------------

def make_ddim_timesteps(num_steps, total_timesteps):
    """
    Create a reduced DDIM timestep sequence.

    Parameters
    ----------
    num_steps : int
        Number of DDIM reverse steps.

    total_timesteps : int
        Total number of original DDPM timesteps.

    Returns
    -------
    reverse_timesteps : list
        A list of timestep indices in descending order.
    """

    assert num_steps >= 2, (
        "num_steps should be at least 2 so that the sequence "
        "contains both a noisy state and a near-clean state."
    )

    assert num_steps <= total_timesteps, (
        "num_steps should not be larger than the total number "
        "of DDPM timesteps."
    )

    # Step 2.1:
    # Create evenly spaced timestep indices from 0 to T - 1.
    # This gives us a smaller DDIM path inside the original DDPM schedule.
    # WriteYourCodeHere
    selected = torch.linspace(
        0,
        total_timesteps - 1,
        steps=num_steps
    ).round().long()

    # Step 2.2:
    # Remove possible duplicates caused by rounding.
    # This keeps the timestep sequence valid.
    # WriteYourCodeHere
    selected = torch.unique(
        selected,
        sorted=True
    )

    # Step 2.3:
    # Make sure timestep 0 is included.
    # Timestep 0 is the closest scheduled state before the final
    # clean estimate.
    if selected[0].item() != 0:
        selected = torch.cat(
            [
                torch.tensor([0], dtype=torch.long),
                selected
            ]
        )

    # Step 2.4:
    # Make sure timestep T - 1 is included.
    # This is the starting point x_T of the reverse process.
    if selected[-1].item() != total_timesteps - 1:
        selected = torch.cat(
            [
                selected,
                torch.tensor([total_timesteps - 1], dtype=torch.long)
            ]
        )

    # Step 2.5:
    # Reverse the order for denoising:
    # T - 1 -> ... -> 0
    # WriteYourCodeHere
    reverse_timesteps = selected.flip(
        dims=[0]
    ).tolist()

    return reverse_timesteps


# ------------------------------------------------------------
# Step 3:
# Implement one deterministic DDIM update.
#
# The DDIM update has two conceptual parts:
#
# 1. Estimate the clean image x0_hat from the current noisy image.
#
# 2. Move from the current timestep t to a previous timestep t_prev
#    using the same predicted noise direction.
#
# For eta = 0, no extra random noise is injected.
# This is the deterministic DDIM setting.
# ------------------------------------------------------------

def ddim_step(
    x_current,
    current_timestep,
    previous_timestep,
    predicted_noise
):
    """
    Perform one deterministic DDIM update.

    Parameters
    ----------
    x_current : torch.Tensor
        Current noisy image x_t.

    current_timestep : int
        Current diffusion timestep t.

    previous_timestep : int
        Previous diffusion timestep t_prev.
        Use -1 to represent the final clean-image state.

    predicted_noise : torch.Tensor
        Predicted noise epsilon_theta(x_t, t).

    Returns
    -------
    x_previous : torch.Tensor
        Image at the previous timestep.

    x0_hat : torch.Tensor
        Estimated clean image from the current timestep.
    """

    current_timestep_tensor = torch.tensor(
        [current_timestep],
        device=device,
        dtype=torch.long
    )

    # Step 3.1:
    # Estimate x0 from the current noisy image and predicted noise.
    #
    # This reuses the clean-image estimation formula from Task 4.
    # WriteYourCodeHere
    x0_hat = predict_x0_from_noise(
        x_noisy=x_current,
        timesteps=current_timestep_tensor,
        predicted_noise=predicted_noise
    )

    # Step 3.2:
    # Read alpha_bar for the previous timestep.
    #
    # If previous_timestep is -1, alpha_bar_previous = 1.
    # WriteYourCodeHere
    alpha_bar_previous = get_alpha_bar_at_timestep(
        previous_timestep
    )

    # Step 3.3:
    # Convert alpha_bar_previous into image-shaped coefficients.
    #
    # These coefficients decide how much clean signal and noise
    # should remain at the previous timestep.
    # WriteYourCodeHere
    previous_signal_coefficient = torch.sqrt(
        alpha_bar_previous
    ).reshape(1, 1, 1, 1)

    # WriteYourCodeHere
    previous_noise_coefficient = torch.sqrt(
        1.0 - alpha_bar_previous
    ).reshape(1, 1, 1, 1)

    # Step 3.4:
    # Apply the deterministic DDIM update:
    #
    # x_t_prev = sqrt(alpha_bar_t_prev) * x0_hat
    #            + sqrt(1 - alpha_bar_t_prev) * predicted_noise
    #
    # WriteYourCodeHere
    x_previous = (
        previous_signal_coefficient * x0_hat
        +
        previous_noise_coefficient * predicted_noise
    )

    return x_previous, x0_hat


# ------------------------------------------------------------
# Step 4:
# Generate the final noisy MRI x_T.
#
# We use the same true Gaussian noise for all DDIM reconstructions.
# This makes the comparison across different step counts fair.
#
# In this controlled experiment, this true noise will also serve
# as the oracle noise prediction during reverse reconstruction.
# ------------------------------------------------------------

final_timestep = T - 1

final_timestep_tensor = torch.tensor(
    [final_timestep],
    device=device,
    dtype=torch.long
)

# Step 4.1:
# Sample one Gaussian noise tensor.
# WriteYourCodeHere
oracle_noise = torch.randn_like(
    x0
)

# Step 4.2:
# Generate x_T from x0 using the DDPM forward process.
# WriteYourCodeHere
x_T, oracle_noise = q_sample(
    x_start=x0,
    timesteps=final_timestep_tensor,
    noise=oracle_noise
)


# ------------------------------------------------------------
# Step 5:
# Define a full DDIM reconstruction function.
#
# This function starts from x_T and repeatedly applies ddim_step().
#
# In a real diffusion model, predicted_noise would come from a
# trained neural network at every reverse step.
#
# Here, predicted_noise is set to oracle_noise.
# This lets us isolate the DDIM update rule itself.
# ------------------------------------------------------------

def ddim_reconstruct_with_oracle(
    x_start_noisy,
    oracle_noise,
    num_steps
):
    """
    Reconstruct x0 from x_T using deterministic DDIM updates.

    Parameters
    ----------
    x_start_noisy : torch.Tensor
        Starting noisy image, usually x_T.

    oracle_noise : torch.Tensor
        True Gaussian noise used in the forward process.

    num_steps : int
        Number of reduced DDIM reverse steps.

    Returns
    -------
    reconstruction : torch.Tensor
        Final reconstructed image.

    trajectory : list
        All intermediate images and timestep labels.
    """

    reverse_timesteps = make_ddim_timesteps(
        num_steps=num_steps,
        total_timesteps=T
    )

    x_current = x_start_noisy.clone()

    trajectory = [
        {
            "timestep": reverse_timesteps[0],
            "image": x_current.detach().clone()
        }
    ]

    for index, current_timestep in enumerate(reverse_timesteps):

        if index + 1 < len(reverse_timesteps):
            previous_timestep = reverse_timesteps[index + 1]
        else:
            previous_timestep = -1

        # Step 5.1:
        # In a real model, this line would be replaced by:
        #
        # predicted_noise = neural_network(x_current, current_timestep)
        #
        # In this controlled experiment, we use the true noise.
        # This is why the reconstruction can be almost exact.
        # WriteYourCodeHere
        predicted_noise = oracle_noise

        # Step 5.2:
        # Move one step along the deterministic DDIM trajectory.
        # WriteYourCodeHere
        x_current, x0_hat = ddim_step(
            x_current=x_current,
            current_timestep=current_timestep,
            previous_timestep=previous_timestep,
            predicted_noise=predicted_noise
        )

        # Step 5.3:
        # Store every intermediate state.
        #
        # This makes it possible to later choose clearer visualization
        # points near the clean image side, such as t = 105, t = 53,
        # and t = 0.
        trajectory.append(
            {
                "timestep": previous_timestep,
                "image": x_current.detach().clone()
            }
        )

    reconstruction = x_current

    return reconstruction, trajectory


# ------------------------------------------------------------
# Step 6:
# Run DDIM reconstruction with different numbers of steps.
#
# Smaller num_steps means fewer reverse updates.
#
# In a real trained diffusion model, this would also mean fewer
# neural network evaluations and faster sampling.
# ------------------------------------------------------------

step_counts = [
    1000,
    100,
    50,
    20,
    10
]

ddim_results = []

for num_steps in step_counts:

    start_time = time.time()

    # WriteYourCodeHere
    reconstruction, trajectory = ddim_reconstruct_with_oracle(
        x_start_noisy=x_T,
        oracle_noise=oracle_noise,
        num_steps=num_steps
    )

    elapsed_time = time.time() - start_time

    mse = torch.mean(
        (reconstruction - x0) ** 2
    ).item()

    mae = torch.mean(
        torch.abs(reconstruction - x0)
    ).item()

    ddim_results.append(
        {
            "num_steps": num_steps,
            "reconstruction": reconstruction,
            "trajectory": trajectory,
            "mse": mse,
            "mae": mae,
            "time": elapsed_time
        }
    )


# ------------------------------------------------------------
# Step 7:
# Checkpoint.
#
# These checks confirm that every DDIM reconstruction has the
# correct shape and valid numerical values.
# ------------------------------------------------------------

for result in ddim_results:

    assert result["reconstruction"].shape == x0.shape, (
        "The DDIM reconstruction shape does not match x0."
    )

    assert torch.isfinite(result["reconstruction"]).all(), (
        "The DDIM reconstruction contains non-finite values."
    )


# ------------------------------------------------------------
# Step 8:
# Print quantitative results.
#
# In this oracle-assisted setting, reconstruction errors should
# be very small because the true noise is used as the prediction.
#
# If 1000 steps has a slightly larger error than 10 steps, this
# does not mean that 10 steps is better in a real model.
# It usually reflects tiny floating-point errors accumulated over
# more update steps.
# ------------------------------------------------------------

print("DDIM reduced-step reconstruction results")
print("----------------------------------------")
print(f"Starting timestep: t = {final_timestep}")
print()

for result in ddim_results:
    print(
        f"{result['num_steps']:4d} steps | "
        f"MSE = {result['mse']:.10f} | "
        f"MAE = {result['mae']:.10f} | "
        f"time = {result['time']:.4f} s"
    )


# ------------------------------------------------------------
# Step 9:
# Visualize x_T and the final DDIM reconstructions.
#
# The image x_T should look like strong Gaussian noise.
# The reconstructions should recover the anatomical MRI structure.
#
# Since this experiment uses oracle noise, all reconstructions may
# look very similar even with fewer steps.
# ------------------------------------------------------------

images_to_show = [
    {
        "title": "Noisy MRI\n$x_T$",
        "image": x_T
    }
]

for result in ddim_results:
    images_to_show.append(
        {
            "title": f"DDIM\n{result['num_steps']} steps",
            "image": result["reconstruction"]
        }
    )


fig, axes = plt.subplots(
    1,
    len(images_to_show),
    figsize=(18, 3.5)
)

for axis, item in zip(axes, images_to_show):

    display_image = (
        denormalize_to_zero_to_one(item["image"])
        .clamp(0, 1)
        .squeeze()
        .detach()
        .cpu()
        .numpy()
    )

    axis.imshow(
        display_image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axis.set_title(
        item["title"]
    )

    axis.axis("off")


plt.suptitle(
    "Oracle-Assisted DDIM Reconstruction with Reduced Steps",
    y=1.05
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Step 10:
# Plot reconstruction MAE versus number of DDIM steps.
#
# Why do we plot MAE instead of MSE here?
#
# In this oracle-assisted experiment, the reconstruction error is
# extremely small because the true noise is provided.
#
# MSE is often too close to zero to be visually informative.
# MAE is easier to interpret in this setting.
# ------------------------------------------------------------

step_values = [
    result["num_steps"] for result in ddim_results
]

mae_values = [
    result["mae"] for result in ddim_results
]

time_values = [
    result["time"] for result in ddim_results
]


plt.figure(figsize=(7, 4))

plt.plot(
    step_values,
    mae_values,
    marker="o"
)

plt.gca().invert_xaxis()

plt.title("DDIM Reconstruction MAE versus Number of Steps")
plt.xlabel("Number of DDIM reverse steps")
plt.ylabel("Mean absolute error")
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Step 11:
# Plot runtime versus number of DDIM steps.
#
# This plot is important because DDIM is mainly useful for
# reducing the number of reverse denoising updates.
#
# In a real trained model, fewer steps usually mean fewer neural
# network evaluations and faster sampling.
# ------------------------------------------------------------

plt.figure(figsize=(7, 4))

plt.plot(
    step_values,
    time_values,
    marker="o"
)

plt.gca().invert_xaxis()

plt.title("Runtime versus Number of DDIM Steps")
plt.xlabel("Number of DDIM reverse steps")
plt.ylabel("Runtime (seconds)")
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# Step 12:
# Visualize one reduced-step DDIM trajectory.
#
# We use the 20-step result because it is short enough to interpret.
#
# Instead of only showing evenly spaced trajectory points, we choose
# more low-timestep states near the clean image side.
#
# This makes the denoising ability clearer because anatomical
# structures are easier to see when t is smaller.
# ------------------------------------------------------------

trajectory_to_show = None

for result in ddim_results:
    if result["num_steps"] == 20:
        trajectory_to_show = result["trajectory"]
        break

assert trajectory_to_show is not None, (
    "The 20-step DDIM trajectory was not found."
)


# ------------------------------------------------------------
# Step 12.1:
# Select representative trajectory points.
#
# t = 999 is the highly noisy starting point.
# t = 263 is still noisy but begins moving toward structure.
# t = 105, 53, and 0 are closer to the clean image side.
# t = -1 is the final clean estimate.
# ------------------------------------------------------------

target_display_timesteps = [
    999,
    263,
    105,
    53,
    0,
    -1
]


selected_trajectory_items = []

for target_timestep in target_display_timesteps:

    closest_item = min(
        trajectory_to_show,
        key=lambda item: abs(
            item["timestep"] - target_timestep
        )
    )

    selected_trajectory_items.append(
        closest_item
    )


# Remove duplicate timestep entries if two targets select the same item.
unique_selected_items = []
seen_timesteps = set()

for item in selected_trajectory_items:

    timestep_value = item["timestep"]

    if timestep_value not in seen_timesteps:
        unique_selected_items.append(item)
        seen_timesteps.add(timestep_value)


# ------------------------------------------------------------
# Step 12.2:
# Display the selected trajectory points.
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    len(unique_selected_items),
    figsize=(16, 3.5)
)

if len(unique_selected_items) == 1:
    axes = [axes]

for axis, item in zip(axes, unique_selected_items):

    display_image = (
        denormalize_to_zero_to_one(item["image"])
        .clamp(0, 1)
        .squeeze()
        .detach()
        .cpu()
        .numpy()
    )

    timestep_label = item["timestep"]

    if timestep_label == -1:
        title = "Final\nclean estimate"
    else:
        title = f"t = {timestep_label}"

    axis.imshow(
        display_image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axis.set_title(title)
    axis.axis("off")


plt.suptitle(
    "DDIM Reconstruction Trajectory Using 20 Steps",
    y=1.05
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Step 13:
# Final interpretation.
# ------------------------------------------------------------

print()
print("Task 5 completed.")
print("DDIM can reconstruct through a reduced timestep sequence.")
print("This experiment used oracle noise prediction, so it demonstrates the DDIM update mechanism rather than real trained-model performance.")
print("The MAE values are very small because the true noise is provided at every step.")
print("In practical diffusion models, fewer DDIM steps reduce model evaluations but may introduce a speed-quality trade-off.")

# Part 7 — Discussion: From DDPM and DDIM to Medical Image Generation

In this notebook, we studied the foundations of diffusion models through a controlled MRI reconstruction experiment.

We started from a clean axial T2-weighted brain MRI image $x_0$, then gradually corrupted it with Gaussian noise using the DDPM forward process.

The forward process can be summarized as:

$$x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\epsilon$$

where $\epsilon \sim \mathcal{N}(0,I)$.

This equation shows that each noisy image $x_t$ is a mixture of the original clean image signal and Gaussian noise.

As the timestep $t$ increases, the clean anatomical signal decreases and the noise component increases.



## What DDPM Teaches Us

DDPM provides a clear probabilistic framework for image generation.

The forward process is fixed and does not need to be learned.

The reverse process is learned by a neural network.

Instead of directly predicting the clean image, DDPM trains the model to predict the Gaussian noise added during the forward process:

$$\epsilon_\theta(x_t,t)$$

The simplified training objective is:

$$\mathcal{L}*{\mathrm{simple}} = \mathbb{E}\left[\left|\epsilon-\epsilon*\theta(x_t,t)\right|_2^2\right]$$

Once the noise is predicted, the clean image can be estimated by rearranging the forward equation:

$$\hat{x}_0 = \frac{x_t-\sqrt{1-\bar{\alpha}*t}\epsilon*\theta(x_t,t)}{\sqrt{\bar{\alpha}_t}}$$

This explains why noise prediction is a central idea in DDPM.

If the predicted noise is accurate, the estimated clean image should be close to the original image.

If the predicted noise is inaccurate, reconstruction errors or artifacts may appear.



## What DDIM Adds

DDIM uses the same general noise-prediction idea, but changes the sampling process.

Standard DDPM sampling may require many sequential reverse steps, often close to the original number of diffusion timesteps.

DDIM allows sampling with a reduced set of timesteps.

Instead of visiting every timestep, DDIM selects a shorter timestep sequence:

$$\tau = {\tau_1,\tau_2,\ldots,\tau_S}, \quad S \ll T$$

This makes sampling faster because fewer reverse updates are needed.

In the deterministic DDIM setting, the update can be written as:

$$x_{t_{\mathrm{prev}}} = \sqrt{\bar{\alpha}*{t*{\mathrm{prev}}}}\hat{x}*0 + \sqrt{1-\bar{\alpha}*{t_{\mathrm{prev}}}}\epsilon_\theta(x_t,t)$$

In this notebook, we used an oracle-assisted DDIM experiment.

This means that the true noise was provided as the predicted noise.

Therefore, the DDIM section demonstrates the deterministic update mechanism and reduced-step trajectory, rather than the performance of a trained medical image generation model.

---

## DDPM versus DDIM

| Method | Main Idea                                                           | Sampling Behavior                         | Key Advantage                   | Limitation                                            |
| ------ | ------------------------------------------------------------------- | ----------------------------------------- | ------------------------------- | ----------------------------------------------------- |
| DDPM   | Learns to reverse a Gaussian noising process                        | Usually uses many small reverse steps     | Strong probabilistic foundation | Sampling can be slow                                  |
| DDIM   | Uses the same noise-prediction model with a different sampling path | Can use fewer deterministic reverse steps | Faster sampling                 | Fewer steps may reduce quality in real trained models |

DDPM helps us understand the basic diffusion mechanism.

DDIM shows how the same model can be sampled more efficiently.

Together, they provide an important foundation for modern image generation methods.



## Medical Imaging Perspective

Medical image generation is different from natural image generation.

For natural images, visual realism is often an important goal.

For medical images, visual realism alone is not sufficient.

A generated medical image may look plausible but still be clinically unreliable.

Possible risks include:

* removing a real lesion;
* introducing a false abnormality;
* changing organ boundaries;
* smoothing diagnostically important texture;
* producing anatomically inconsistent structures.

Therefore, medical image generation should not be evaluated only by visual quality.

It should also consider anatomical fidelity, pathology preservation, quantitative consistency, and clinical relevance.

A visually realistic medical image is not necessarily a clinically faithful image.



## Limitations of This Notebook

This notebook is designed as a foundational tutorial.

It does not train a full diffusion model on a medical imaging dataset.

It does not evaluate clinical performance.

It does not produce validated synthetic medical images.

Instead, it focuses on the core computational mechanisms:

* how Gaussian noise is added in DDPM;
* how the noise schedule controls signal corruption;
* how noisy MRI images can be sampled at arbitrary timesteps;
* why DDPM predicts noise;
* how a clean image estimate can be recovered from predicted noise;
* how DDIM reduces the number of reverse sampling steps.

The DDIM reconstruction experiment used oracle noise prediction.

This is useful for understanding the mathematics, but it should not be interpreted as evidence that a real trained DDIM model can reconstruct medical images perfectly with very few steps.



## Connection to Flow Matching and SB-CFM

DDPM and DDIM describe image generation as a process of moving between noise and data.

DDPM uses a stochastic denoising process.

DDIM introduces a deterministic sampling path.

Flow Matching and Conditional Flow Matching extend this perspective by learning continuous transformations between distributions.

Instead of explicitly following a long discrete denoising chain, Flow Matching methods learn a velocity field that transports samples from a source distribution to a target distribution.

This creates a natural connection to Schrödinger Bridge Conditional Flow Matching.

In the next notebook, SB-CFM can be understood as a more advanced framework for learning conditional generative paths, especially when the goal is to model structured transformations between medical image distributions.

The concepts from this notebook provide the foundation for that transition:

* $x_0$ as clean data;
* Gaussian noise as a source distribution;
* timestep-dependent transformations;
* learned prediction functions;
* reconstruction paths from noise to image;
* speed-quality trade-offs in sampling.

---

## Summary

This notebook introduced the core ideas behind DDPM and DDIM using a controlled medical imaging example.

We prepared a clean MRI image, applied DDPM forward noising, visualized the loss of anatomical structure, estimated clean images from predicted noise, and implemented reduced-step DDIM reconstruction.

The main takeaway is:

Diffusion models learn how to move between clean data and noise.

DDPM explains how this process can be formulated as gradual denoising.

DDIM shows how the reverse path can be accelerated.

These ideas are essential for understanding more advanced generative methods in medical imaging, including Flow Matching and SB-CFM.
